<a href="https://colab.research.google.com/github/OTogano/Bioinformatics-Project/blob/main/notebook/BRCA_Tumor_vs_Normal_Expression_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Breast Tumor vs Normal Tissue: Gene Expression Analysis and Classification

## Project overview

**Research question:** Which genes and pathways differ in expression between breast tumor and normal tissue, and can those expression differences accurately classify samples?

This project uses gene expression data from breast cancer patients to find genes that behave differently in tumor tissue compared to normal breast tissue, to interpret what those differences mean biologically, and to build machine learning models that classify a sample as tumor or normal.

## Group members

- Clyde Mugambi - 166330
- Mohamed Chaudhry - 166335
- Mark Mutinda - 150066
- Matthew Wachira - 151733
- Lee Gitonga - 150465

## Dataset

**TCGA-BRCA** (The Cancer Genome Atlas, breast cancer cohort), obtained from the UCSC Xena browser (GDC hub): STAR gene expression counts, a phenotype table with sample information, and a gene ID map.

## Project stages

1. Question and objectives
2. Data acquisition
3. Pre-processing and quality control
4. Exploratory data analysis
5. Core analysis (differential expression)
6. Biological interpretation
7. Machine learning (logistic regression and random forest)
8. Conclusions

## About this notebook

This notebook contains the full analysis for the project, organised by stage. Each stage has its own section heading.

Stages 2 and 3 produce the cleaned data (`X_clean.csv.gz` and related files) saved in the shared Drive `processed` folder, which the later stages load.

## Stage 2: Data Acquisition

### Step 1: Connect to Google Drive and set the data folders

The group's data lives in a shared Google Drive folder, so every member loads the same files. This cell connects Colab to Drive and defines two folder paths:

- `RAW`: the original files downloaded from UCSC Xena (TCGA-BRCA from the GDC hub): gene expression counts, the phenotype table and the gene ID map.
- `PROCESSED`: where the cleaned data is saved at the end of Stage 3, ready for the analysis and modelling stages.

**Note:** If you are running this notebook locally, then make sure to download the dataset files and place it under a new folder in the root of this project called `dataset/`. Your folder structure should look like this:

```bash
.
└── dataset/
    ├── processed/
    │   ├── gene_map_clean.csv
    │   ├── sample_metadata.csv
    │   ├── X_clean_csv.gz
    │   └── y_labels.csv
    ├── raw/
    │   ├── genecode.v36.annotation.gft.gene.probemap
    │   ├── TCGA-BRCA.clinical.tsv.gz
    │   └── TCGA-BRCA.star_counts.tsv.gz
    ├── notebook/
    │   └── BRCA_TUmor_vs_Normal_Expression_Analysis.ipynb
    ├── .gitignore
    ├── README.md
    └── SETUP.md
```


In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAW = '/content/drive/MyDrive/Bioinformatics Project Dataset/raw/'
    PROCESSED = '/content/drive/MyDrive/Bioinformatics Project Dataset/processed/'
except ModuleNotFoundError:
    RAW = '../dataset/raw/'
    PROCESSED = '../dataset/processed/'


### Step 2: Load the three data files

This cell loads the three raw files into pandas, one table (dataframe) for each:

- `expr`: the **gene expression table**. Each row is a gene and each column is a sample. The values are log2(count + 1), so a value of 0 means the gene had no reads in that sample. It is stored as `float32` to save memory, because it holds about 60,660 genes.
- `pheno`: the **phenotype table**, with one row per sample and 84 columns of sample information, including whether the sample is a tumor or normal tissue.
- `gmap`: the **gene map**, which links each Ensembl gene ID (such as `ENSG00000000003`) to its gene name (such as `TSPAN6`).

The printed shapes (rows, columns) show how much data was loaded, and the previews and column list let us check that the files read correctly before we use them.

In [3]:
import numpy as np
import pandas as pd

# 1. Expression: genes x samples (large, so we use float 32 instead)
expr = pd.read_csv(RAW + "TCGA-BRCA.star_counts.tsv.gz", sep="\t", index_col=0, dtype = {0: str})
expr = expr.astype("float32")
print("expr:", expr.shape)

# 2. Phenotype: one row per sample
pheno = pd.read_csv(RAW + "TCGA-BRCA.clinical.tsv.gz", sep="\t", index_col=0)
print("pheno:", pheno.shape)

# 3. Gene mapping: Ensembl ID -> gene name
gmap = pd.read_csv(RAW + "gencode.v36.annotation.gtf.gene.probemap", sep="\t")
print("gmap:", gmap.shape)

# Look at each one
display(expr.iloc[:5, :5])
display(pheno.head())
display(gmap.head())
print(pheno.columns.tolist())


expr: (60660, 1226)
pheno: (1255, 84)
gmap: (60660, 6)


,TCGA-D8-A146-01A,TCGA-AQ-A0Y5-01A,TCGA-C8-A274-01A,TCGA-BH-A0BD-01A,TCGA-B6-A1KC-01B
Ensembl_ID,,,,,
ENSG00000000003.15,11.737670,9.781360,13.122504,11.016809,11.000000
ENSG00000000005.6,7.721099,3.321928,0.000000,6.686501,3.807355
ENSG00000000419.13,11.042343,11.357552,11.506308,10.801708,11.074142
ENSG00000000457.14,11.036860,10.754888,12.218261,11.190442,10.857981
ENSG00000000460.17,9.131857,8.721099,10.973698,10.761551,9.550747


,id,disease_type,case_id,submitter_id,primary_site,alcohol_history.exposures,race.demographic,gender.demographic,ethnicity.demographic,vital_status.demographic,...,days_to_collection.samples,initial_weight.samples,preservation_method.samples,pathology_report_uuid.samples,oct_embedded.samples,specimen_type.samples,days_to_sample_procurement.samples,is_ffpe.samples,tissue_type.samples,annotations.samples
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-BH-A0W3-01A,3c612e12-6de8-44fa-a095-805c45474821,Ductal and Lobular Neoplasms,3c612e12-6de8-44fa-a095-805c45474821,TCGA-BH-A0W3,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,85.0,120.0,OCT,801A4E2F-E26E-424F-BF42-CD0D9CD62BCE,True,Solid Tissue,NaN,False,Tumor,NaN
TCGA-AR-A24V-01A,3cb06c7a-f2a8-448b-91a8-dd201bbf2ddd,Ductal and Lobular Neoplasms,3cb06c7a-f2a8-448b-91a8-dd201bbf2ddd,TCGA-AR-A24V,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,1720.0,400.0,OCT,468CD293-C9F7-43C6-A40A-18FCDD22F6AA,True,Solid Tissue,NaN,False,Tumor,NaN
TCGA-E9-A1NE-01A,3d676bba-154b-4d22-ab59-d4d4da051b94,Ductal and Lobular Neoplasms,3d676bba-154b-4d22-ab59-d4d4da051b94,TCGA-E9-A1NE,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,31.0,280.0,OCT,CF6E29A2-FAE6-45BB-B625-33877887A89E,True,Solid Tissue,NaN,False,Tumor,NaN
TCGA-E9-A1NE-11A,3d676bba-154b-4d22-ab59-d4d4da051b94,Ductal and Lobular Neoplasms,3d676bba-154b-4d22-ab59-d4d4da051b94,TCGA-E9-A1NE,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,31.0,830.0,OCT,NaN,True,Solid Tissue,NaN,False,Normal,NaN
TCGA-AC-A8OQ-01A,dfaabd03-2d40-4422-b210-caf112ff4229,Ductal and Lobular Neoplasms,dfaabd03-2d40-4422-b210-caf112ff4229,TCGA-AC-A8OQ,Breast,Not Reported,black or african american,female,not hispanic or latino,Alive,...,742.0,100.0,Unknown,FFA6F9F3-71C1-4AF9-B9F7-0466550EBC90,False,Solid Tissue,NaN,False,Tumor,NaN


,id,gene,chrom,chromStart,chromEnd,strand
0,ENSG00000223972.5,DDX11L1,chr1,11869,14409,+
1,ENSG00000227232.5,WASH7P,chr1,14404,29570,-
2,ENSG00000278267.1,MIR6859-1,chr1,17369,17436,-
3,ENSG00000243485.5,MIR1302-2HG,chr1,29554,31109,+
4,ENSG00000284332.1,MIR1302-2,chr1,30366,30503,+


['id', 'disease_type', 'case_id', 'submitter_id', 'primary_site', 'alcohol_history.exposures', 'race.demographic', 'gender.demographic', 'ethnicity.demographic', 'vital_status.demographic', 'age_at_index.demographic', 'days_to_birth.demographic', 'year_of_birth.demographic', 'year_of_death.demographic', 'primary_site.project', 'project_id.project', 'disease_type.project', 'name.project', 'name.program.project', 'tissue_source_site_id.tissue_source_site', 'code.tissue_source_site', 'name.tissue_source_site', 'project.tissue_source_site', 'bcr_id.tissue_source_site', 'days_to_death.demographic', 'entity_submitter_id.annotations', 'notes.annotations', 'submitter_id.annotations', 'classification.annotations', 'entity_id.annotations', 'created_datetime.annotations', 'annotation_id.annotations', 'entity_type.annotations', 'updated_datetime.annotations', 'case_id.annotations', 'state.annotations', 'category.annotations', 'status.annotations', 'case_submitter_id.annotations', 'synchronous_mali

### Step 3: Match the samples and check the labels

This cell links the expression table to the phenotype table and checks what kinds of samples we have.

- **Matching samples:** the sample ID (for example `TCGA-E9-A1NE-11A`) is the link between the two tables. We keep only the samples found in both, and store their tumor/normal information in `meta`. All 1,226 expression samples had a match.
- **Label check:** the two count tables show how many samples are Tumor or Normal (1,113 and 113), and the finer sample types. The finer types reveal that 7 of the "tumor" samples are **metastatic** (cancer that spread from elsewhere), so we handle them in the next step.
- **Gene name check:** we confirm that every gene ID in the expression table appears in the gene map. We also count gene names shared by more than one Ensembl ID (1,233). This matters later, so we keep the Ensembl IDs as the unique gene identifiers.

In [4]:
# Keep only samples present in both tables
common = expr.columns.intersection(pheno.index)
print("samples in both: ",len(common))

meta = pheno.loc[common, ["tissue_type.samples", "sample_type.samples", "submitter_id"]]
print(meta["tissue_type.samples"].value_counts(dropna=False))
print(meta["sample_type.samples"].value_counts(dropna=False))



samples in both:  1226
tissue_type.samples
Tumor     1113
Normal     113
Name: count, dtype: int64
sample_type.samples
Primary Tumor          1106
Solid Tissue Normal     113
Metastatic                7
Name: count, dtype: int64


## Stage 3: Pre-processing and Quality Control

### Step 1: Keep primary tumor and normal samples, and build the labels

This cell starts the cleaning. Because the research question compares **breast tumor with normal breast tissue**, we keep only two sample types: *Primary Tumor* and *Solid Tissue Normal*. The 7 metastatic samples are dropped, since they come from cancer that spread from elsewhere. That leaves 1,219 samples.

- **Labels (`y`):** each sample gets a label, 1 for tumor and 0 for normal. This is the target the machine learning models will predict. The counts show how many of each we have (1,106 tumor and 113 normal), which reveals a strong **class imbalance**.
- **Expression table (`X`):** the expression data is flipped so that each **row is a sample** and each **column is a gene**, the layout that scikit-learn expects.
- **Patient check:** the last lines count how many patients have more than one sample of the same type. There are 1,095 unique patients, and 11 of them have two tumor samples (duplicate vials from the same tumor). The normal samples have no repeats, but each one is paired with a tumor sample from the same patient. We handle the duplicate tumor samples in the next step.

In [5]:
# Keep primary tumor and normal only
keep = meta[meta["sample_type.samples"].isin(["Primary Tumor", "Solid Tissue Normal"])]
print("kept:", keep.shape[0], "| dropped:", meta.shape[0] - keep.shape[0])

# Labels: 1 = tumor, 0 = normal
y = (keep["sample_type.samples"] == "Primary Tumor").astype(int)
print(y.value_counts())

# Expression: samples as rows, genes as columns
X = expr[keep.index].T
print("X:", X.shape)

# Do any patients appear more than once?
print("unique patients:", keep["submitter_id"].nunique())
print(keep.groupby("sample_type.samples")["submitter_id"].apply(lambda s: s.duplicated().sum()))

kept: 1219 | dropped: 7
sample_type.samples
1    1106
0     113
Name: count, dtype: int64
X: (1219, 60660)
unique patients: 1095
sample_type.samples
Primary Tumor          11
Solid Tissue Normal     0
Name: submitter_id, dtype: int64


### Step 2: Remove duplicate tumor samples (one sample per patient)

Some patients have **two tumor samples** (for example `-01A` and `-01B`, two vials from the same tumor). These are almost identical, so if one landed in the training set and the other in the test set, the model would effectively see the answer in advance and its accuracy would look better than it really is.

This cell keeps **one sample per patient and sample type** (the first one, `-01A`) and drops the 11 repeats. It then rebuilds the labels `y` and the expression table `X` for the remaining samples. The `patient` column is kept so that later stages can split the data by patient.

The printed output confirms the result: **1,208 samples** (1,095 tumor and 113 normal) with 60,660 genes each.

In [6]:
keep = keep.sort_index()
keep["patient"] = keep["submitter_id"]
keep = keep.drop_duplicates(subset=["patient", "sample_type.samples"], keep="first")

y = (keep["sample_type.samples"] == "Primary Tumor").astype(int)
X = expr[keep.index].T
print("X:", X.shape)
print(y.value_counts())

X: (1208, 60660)
sample_type.samples
1    1095
0     113
Name: count, dtype: int64


### Step 3: Quality control checks

Before filtering anything, this cell checks the health of the expression table `X` and records its size (`samples_before`, `genes_before`) so we can report counts before and after cleaning.

- **Missing values and negative values:** both are 0. Expression values cannot be negative, so this confirms the data is valid.
- **Duplicate samples and duplicate genes:** no sample is repeated. 2,693 genes are duplicates of other genes, mostly because they are zero in every sample and so look identical to each other.
- **Zero expression:** 2,640 genes are zero in every sample, 16,540 are zero in at least 90% of samples, and 29,051 are zero in at least 50%. This is normal for RNA-seq, because many of the 60,660 annotated genes are not active in breast tissue. These genes carry no useful signal, which is why we filter them in the next step.
- **Value range:** the values run from 0 to about 22.9 with a median near 10.5, which fits log2-transformed counts, so no further transformation is needed.

In [7]:
genes_before, samples_before = X.shape[1], X.shape[0]

print("missing values:", int(X.isna().sum().sum()))
print("negative values:", int((X < 0).sum().sum()))
print("duplicate samples:", int(X.duplicated().sum()))
print("duplicate genes:", int(X.T.duplicated().sum()))

zero_frac = (X == 0).mean()
print("genes zero in ALL samples:", int((zero_frac == 1).sum()))
print("genes zero in >=90% of samples:", int((zero_frac >= 0.9).sum()))
print("genes zero in >=50% of samples:", int((zero_frac >= 0.5).sum()))
print("overall min/median/max:", float(X.min().min()),
      float(np.median(X.values[:, :2000])), float(X.max().max()))

missing values: 0
negative values: 0
duplicate samples: 0
duplicate genes: 2693
genes zero in ALL samples: 2640
genes zero in >=90% of samples: 16540
genes zero in >=50% of samples: 29051
overall min/median/max: 0.0 10.51865291595459 22.87331199645996


### Step 4: Compare gene filtering thresholds

Many genes are barely expressed, so they add noise and slow down later analysis without adding information. Before removing any, this cell tests how many genes would remain under three different rules.

A gene counts as **expressed** in a sample if it has at least 10 reads. On the log2 scale used in this data, that is `log2(10 + 1) ≈ 3.46`. The cell then asks: in what share of samples is each gene expressed?

| Gene must be expressed in at least | Genes kept |
|---|---|
| 5% of samples | 28,342 |
| 10% of samples | 26,054 |
| 20% of samples | 23,697 |

We choose the **5% rule** in the next step. Only 113 of the 1,208 samples (about 9%) are normal tissue, so a stricter rule could remove genes that are active mainly in normal tissue. This filter uses only expression values, never the tumor/normal labels, so it does not leak information about the answer.

In [8]:
thr = np.log2(10 + 1)   # count >= 10

for frac in [0.05, 0.10, 0.20]:
    n_keep = int(((X >= thr).mean() >= frac).sum())
    print(f"expressed (count>=10) in >= {int(frac*100)}% of samples: {n_keep} genes kept")

expressed (count>=10) in >= 5% of samples: 28342 genes kept
expressed (count>=10) in >= 10% of samples: 26054 genes kept
expressed (count>=10) in >= 20% of samples: 23697 genes kept


### Step 5: Filter out low-expression genes

This cell applies the 5% rule chosen in the previous step: a gene is kept only if it has at least 10 reads in at least 5% of the samples. The remaining genes are stored in `X_clean`, the cleaned expression table.

The printed summary shows the effect:

- **Samples:** 1,208 → 1,208. Gene filtering does not remove any samples. The sample cleaning was done in Steps 1 and 2 of this stage.
- **Genes:** 60,660 → **28,342**. About half the genes were not meaningfully expressed in breast tissue and are removed.
- **Checks on the result:** no all-zero genes, no duplicate genes and no missing values remain, so the table is clean.

Data is **not scaled** here. The values are already log2-normalized counts, and scaling should be done later inside each model, fitted on the training data only, to avoid leaking information from the test set.

In [9]:
thr = np.log2(10 + 1)
mask = (X >= thr).mean() >= 0.05
X_clean = X.loc[:, mask]

print(f"Samples: {samples_before} -> {X_clean.shape[0]}")
print(f"Genes:   {genes_before} -> {X_clean.shape[1]}")
print("zero-in-all genes left:", int((X_clean == 0).all().sum()))
print("duplicate genes left:", int(X_clean.T.duplicated().sum()))
print("missing values:", int(X_clean.isna().sum().sum()))

Samples: 1208 -> 1208
Genes:   60660 -> 28342
zero-in-all genes left: 0
duplicate genes left: 0
missing values: 0


### Step 6: Save the cleaned data for the group

This cell saves the cleaned data to the `processed` folder in the shared Drive, so the other stages (exploratory analysis, differential expression, biological interpretation and the machine learning models) all start from the same data. Four files are written:

- `X_clean.csv.gz`: the cleaned expression table, 1,208 samples × 28,342 genes (compressed to save space).
- `y_labels.csv`: the label for each sample, 1 for tumor and 0 for normal.
- `sample_metadata.csv`: the patient ID and sample type for each sample. This lets the modelling stage split the data **by patient**, so the same patient never appears in both the training and test sets.
- `gene_map_clean.csv`: the Ensembl ID to gene name map, kept only for the genes that remain, for use in the biological interpretation stage.

The printed list confirms all four files were saved.

In [10]:
import os

# Don't save the files again if they already exist
save_data: bool = False
try:
    save_data = (not os.path.exists(PROCESSED)) or (len(os.listdir(PROCESSED)) == 0)
except:
    save_data = True

if (save_data):
    os.makedirs(PROCESSED, exist_ok=True)

    X_clean.to_csv(PROCESSED + "X_clean.csv.gz", float_format="%.4f", compression="gzip")
    y.rename("label").to_csv(PROCESSED + "y_labels.csv")
    keep[["patient", "sample_type.samples"]].to_csv(PROCESSED + "sample_metadata.csv")
    gmap[gmap["id"].isin(X_clean.columns)].to_csv(PROCESSED + "gene_map_clean.csv", index=False)
    print(os.listdir(PROCESSED))

## Summary of Stages 2 and 3

| Step | Samples | Genes |
|---|---|---|
| Raw expression file | 1,226 | 60,660 |
| After removing 7 metastatic samples | 1,219 | 60,660 |
| After removing 11 duplicate tumor samples | 1,208 (1,095 tumor, 113 normal) | 60,660 |
| After filtering low-expression genes | 1,208 | 28,342 |

**Notes for the next stages**

- The data is log2(count + 1) normalized and **not scaled**. Fit any scaling on the training data only.
- There is a strong **class imbalance** (113 normal against 1,095 tumor). Use stratified splits and class weights, and report metrics beyond accuracy.
- The 113 normal samples are **paired** with tumor samples from the same patient. Split by patient using `sample_metadata.csv`.
- 1,233 gene names are shared by more than one Ensembl ID. Ensembl IDs are the unique gene keys, and duplicate names should be handled when gene names are needed for interpretation.

The cleaned data is saved in the shared Drive folder `processed`: `X_clean.csv.gz`, `y_labels.csv`, `sample_metadata.csv` and `gene_map_clean.csv`.

## Stage 4: Exploratory Data Analysis

## Stage 5: Core Analysis

## Stage 6: Biological Interpretation

## Stage 7: Machine Learning Modeling and Prediction
We chose two machine learning classification models to use for making predicitions based on the dataset, which are:
1. Logistic Regression
2. Random Forest

### Step 1: Create the train-test split for the data
The first part is to make the train-test split of the data. In this step, we need to handle the class imbalance when spliting the data, otherwise the scores won't be accurate. Furthermore, we need to make sure that the same patient is grouped together with its sample in the split. If the patient has two or more samples that are both normal and cancerous tissues, it'll mess up the model's predictions.

To solve this, we can use scikit-learn's `StratifiedGroupKFold` object to create the train-test split: it handles both the class imbalance as well as ensuring the same patient samples are grouped together.

In [11]:
from sklearn.model_selection import StratifiedGroupKFold

# Get all patients in the table to group by
groups = keep.loc[X_clean.index, "patient"]

# Create the stratified group fold.
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
# Get the indexes for the training and testing samples
train_idx, test_idx = next(sgkf.split(X_clean, y, groups))

X_train, X_test = X_clean.iloc[train_idx], X_clean.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
# We need to also split the groups table as it'll be used during cross-validation
g_train = groups.iloc[train_idx]

print("train:", X_train.shape, y_train.value_counts().to_dict())
print("test: ", X_test.shape,  y_test.value_counts().to_dict())

train: (967, 28342) {1: 876, 0: 91}
test:  (241, 28342) {1: 219, 0: 22}


### Logistic Regression
With the data now split, we can create a pipeline to transform the data and fit it to a logistic regression model.

For transforming the data, we'll first do feature selection to use the best representing features/genes present in the sample for modeling. Once the features have been selected, we can normalize the data.

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Create the pipeline with each of the transformation stages, and the final model at the end
lr = Pipeline([
    ("select", SelectKBest(f_classif, k=2000)),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=5000)),
])

# Use cross-validation to fit the model; this should allow for the class imbalance to be accounted for during training.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
# Get the score from training
cv_score = cross_val_score(lr, X_train, y_train, groups=g_train, cv=cv, scoring="roc_auc")
print(f"Logistic Regression Score: {cv_score.mean()}")

lr.fit(X_train, y_train)

Logistic Regression Score: 0.9998766993240678


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('select', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](28342,)","['ENSG00000000003.15','ENSG00000000005.6','ENSG00000000419.13',..., 'ENSG00000288670.1','ENSG00000288674.1','ENSG00000288675.1']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,28342
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",2000
,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_c...x7fe2920303b0>
Name,Type,Value


### Random Forest
The random forest model is trained below in a similar way to logistic regression.

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500, class_weight="balanced_subsample",
    n_jobs=-1, random_state=42,
)
cv_score = cross_val_score(rf, X_train, y_train, groups=g_train, cv=cv, scoring="roc_auc")
print(f"Random Forest Score: {cv_score.mean()}")

rf.fit(X_train, y_train)

Random Forest Score: 0.9992454621401989


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced_subsample'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consid

### Model Evaluation
With the models now trained and fitted to the data, we can evaluate how accurate they are.

The models are evaluated using the following metrics:
- The ROC score,
- Balanced accuracy score,
- Confusion matrix to get the true and false positives and negatives,
- Classification report

In [16]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, balanced_accuracy_score)

for name, model in [("Logistic Regression", lr), ("Random Forest", rf)]:
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    print(f"\n=== {name} ===")
    print("AUROC:", round(roc_auc_score(y_test, prob), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_test, pred), 4))
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, target_names=["normal", "tumor"]))


=== Logistic Regression ===
AUROC: 0.9998
Balanced accuracy: 0.9977
[[ 22   0]
 [  1 218]]
              precision    recall  f1-score   support

      normal       0.96      1.00      0.98        22
       tumor       1.00      1.00      1.00       219

    accuracy                           1.00       241
   macro avg       0.98      1.00      0.99       241
weighted avg       1.00      1.00      1.00       241


=== Random Forest ===
AUROC: 0.9985
Balanced accuracy: 0.9091
[[ 18   4]
 [  0 219]]
              precision    recall  f1-score   support

      normal       1.00      0.82      0.90        22
       tumor       0.98      1.00      0.99       219

    accuracy                           0.98       241
   macro avg       0.99      0.91      0.95       241
weighted avg       0.98      0.98      0.98       241

